In [1]:
# ============================================================
# PART 0 — Setup (Kaggle)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # before `import torch`

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn

sns.set_style("whitegrid")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

!pip install pennylane pennylane-lightning --upgrade -q

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}  device: {DEVICE}")

# --- EDIT THESE PATHS FOR YOUR KAGGLE SESSION ---------------------
DATASET = "CICIoT2023"
SCRIPTS_PARENT_DIR = "/kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts"         
DATA_DIR           = f"/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/FROZEN/{DATASET}"
CHECKPOINT_PATH    = "/kaggle/input/models/lawunnannda/qsentinel-models/pytorch/default/9/final-ciciot2023-vqc-train-checkpoint.pt"
HANDOFF_PATH = Path("/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/team-artifacts/teamC_week3_FROZEN_handoff.json")
# --------------------------------------------------------------------------


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.9 MB/s eta 0:00:00
CUDA available: False  device: cpu


In [2]:
if SCRIPTS_PARENT_DIR not in sys.path:
    sys.path.append(SCRIPTS_PARENT_DIR)
import scripts

print("scripts loaded from:", scripts.__file__)

from sklearn.metrics import classification_report, confusion_matrix

from scripts.cache import (
    ForwardCache,
    cached_calibrate_threshold,
    cached_conformal_alpha_sweep,
    cached_f_max,
    cached_lipschitz_percentile,
    cached_predict_labels,
    cached_qsnet_infer,
)
from scripts.circuit import build_forward_circuit, create_quantum_device
from scripts.conformal import (
    class_conditional_calibrate,
    min_calibration_size,
    per_class_empirical_far,
)
from scripts.constants import (
    DEFAULT_ALPHA,
    DEFAULT_BETA,
    DEFAULT_CF,
    DEFAULT_LOWER_PERCENTILE,
    DEFAULT_NOISE_RATE,
    DEFAULT_REUPLOAD,
    DEFAULT_UPPER_PERCENTILE,
    PROP1_RESIDUAL_TOL,
    ZERO_DAY,
)
from scripts.data import (
    capped_sample,
    greedy_dpp_sample,
    load_split,
    plot_class_balance_bars,
    plot_class_balance_pie,
    to_angles,
)
from scripts.logging import to_jsonable
from scripts.memory import (
    run_batched_safely,
    safe_empty_cache,
)
from scripts.quantum_metrics import (
    fidelity_pairwise,
)
from scripts.theory import (
    analytic_lipschitz_bound,
    assert_proposition1,
    check_lipschitz_tightness,
    check_proposition1_real_data,
    proposition2_epsilon_beta,
    proposition2_epsilon_robust,
    proposition2_epsilon_star,
    two_sample_discriminability_auroc,
    verify_fidelity_convention,
    worst_case_f_in,
)

# --- Kaggle output layout ---------------------------------------------------
OUT_ROOT   = Path("/kaggle/working/quantum-sentinel")
CACHE_DIR  = OUT_ROOT / "caches"
CKPT_DIR   = OUT_ROOT / "checkpoints"
LOG_DIR    = OUT_ROOT / "logs"
FIG_DIR    = OUT_ROOT / "figures"
TABLE_DIR  = OUT_ROOT / "tables"
for d in (CACHE_DIR, CKPT_DIR, LOG_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  saved figure -> {p}")
    plt.close()

def savejson(obj, name, subdir=LOG_DIR):
    p = Path(subdir) / f"{name}.json"
    with open(p, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)
    print(f"  saved json -> {p}")
    return p

def savecsv(df, name, subdir=TABLE_DIR):
    p = Path(subdir) / f"{name}.csv"
    df.to_csv(p, index=False)
    print(f"  saved table -> {p}")
    return p

check = verify_fidelity_convention(dim=4, n_trials=30, seed=SEED)
print("Fidelity-convention self-check PASSED (synthetic unit test only):", check)

scripts loaded from: /kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts/scripts/__init__.py
Fidelity-convention self-check PASSED (synthetic unit test only): {'max_pairwise_disagreement': 1.5543122344752192e-15, 'max_fvg_violation': 0.0}


In [3]:
# ============================================================
# PART 1 — Load checkpoint (theta_star, head, prototypes, config)
# ============================================================
print("="*60); print("LOADING CHECKPOINT"); print("="*60)
assert Path(CHECKPOINT_PATH).exists(), f"Not found: {CHECKPOINT_PATH}"
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

theta_star      = ckpt["theta"]
head_state_dict = ckpt["head_state_dict"]
prototypes_raw  = ckpt["prototypes"]
class_names     = ckpt["class_names"]
num_classes     = ckpt["num_classes"]
num_qubits      = ckpt["num_qubits"]
num_layers      = ckpt["num_layers"]
noise_rate      = ckpt.get("noise_rate", DEFAULT_NOISE_RATE)
reupload        = ckpt.get("reupload", DEFAULT_REUPLOAD)
feature_cols    = ckpt["feature_cols"]
use_pca         = ckpt.get("use_pca", True)
target_col      = "label_multiclass"

scaler   = ckpt["scaler"]
pca      = ckpt.get("pca", None)
angle_x_min = np.asarray(ckpt["angle_x_min"])
angle_x_max = np.asarray(ckpt["angle_x_max"])
angle_max   = ckpt.get("angle_max", float(np.pi))

SUBSET        = ckpt.get("subset", True)
PER_CLASS_CAP = ckpt.get("per_class_cap", 7000)
SAMPLER       = ckpt.get("sampler", "greedy_dpp")

prototypes = {int(k): (v if torch.is_tensor(v) else torch.tensor(v)).to(DEVICE)
              for k, v in prototypes_raw.items()}

print(f" classes    : {class_names}")
print(f" qubits     : {num_qubits}  layers: {num_layers}  reupload: {reupload}  noise: {noise_rate}")
print(f" theta      : {tuple(theta_star.shape)}")
print(f" prototypes : {len(prototypes)} classes")

classifier_head = nn.Linear(num_qubits, num_classes).to(DEVICE)
classifier_head.load_state_dict(head_state_dict)
classifier_head.eval()
print(f" head       : Linear({num_qubits} -> {num_classes}) loaded OK")

LOADING CHECKPOINT
 classes    : ['Backdoor_Malware', 'BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PingSweep', 'Recon-PortScan', 'SqlInjection', 'Uploading_Attack', 'VulnerabilityScan', 'XSS']
 qubits     : 6  layers: 3  reupload: True  noise: 0.01
 theta      : (3, 6, 3)
 prototypes : 31 classes
 head       : Linear(6 -> 31) loaded OK


In [4]:
# ============================================================
# Verify Team C's FROZEN features
# ============================================================
print("="*60); print("VERIFYING FROZEN FEATURES"); print("="*60)

handoff = json.loads(HANDOFF_PATH.read_text())
teamc_cols = list(handoff["frozen_subsets"][DATASET]["features"])
ckpt_cols = list(ckpt["feature_cols"])
print(f"Team C selector: {handoff['frozen_subsets'][DATASET]['selector']}\n")
print(f"Team C k={len(teamc_cols)}: {teamc_cols}\n")
print(f"ckpt   k={len(ckpt_cols)}: {ckpt_cols}\n")

same_set = set(teamc_cols) == set(ckpt_cols)
same_order = teamc_cols == ckpt_cols
print(f"same features (set):   {same_set}")
print(f"same order (list):     {same_order}")
print()
if not same_set:
    only_teamc = sorted(set(teamc_cols) - set(ckpt_cols))
    only_ckpt = sorted(set(ckpt_cols) - set(teamc_cols))
    print(f"  only in Team C: {only_teamc}")
    print(f"  only in ckpt:   {only_ckpt}")
    raise ValueError("checkpoint feature_cols do not match Team C FROZEN subset")
if not same_order:
    print("warning: same columns, different order — encoding may still break if order mattered at train time")

VERIFYING FROZEN FEATURES
Team C selector: MI

Team C k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'iat', 'protocol', 'conn_state']

ckpt   k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'iat', 'protocol', 'conn_state']

same features (set):   True
same order (list):     True



In [5]:
# ============================================================
# PART 2 — Load raw splits, apply the SAME subsampling/encoding as training
# ============================================================
print("="*60); print("LOADING RAW DATA SPLITS"); print("="*60)

X_train_full, y_train_full, _ = load_split(DATA_DIR, "train", target_col, csv=True, selected_cols=feature_cols)
X_test,       y_test,       _ = load_split(DATA_DIR, "test", target_col, csv=True, selected_cols=feature_cols)
X_cal,        y_cal,        _ = load_split(DATA_DIR, "calibration", target_col, csv=True, selected_cols=feature_cols)
X_zeroday,    y_zeroday,    _ = load_split(DATA_DIR, "zeroday", target_col, csv=True, selected_cols=feature_cols)

print(f" train(full): {X_train_full.shape}  test: {X_test.shape}  cal: {X_cal.shape}  zeroday: {X_zeroday.shape}")

if SUBSET and SAMPLER == "greedy_dpp":
    X_train, y_train = greedy_dpp_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
elif SUBSET:
    X_train, y_train = capped_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
else:
    X_train, y_train = X_train_full, y_train_full
print(f" train(subset): {X_train.shape} via {SAMPLER}")

def encode(X):
    return to_angles(X, scaler, angle_x_min, angle_x_max,
                      pca=pca if use_pca else None, angle_max=angle_max)

A_train, A_test, A_cal, A_zeroday = encode(X_train), encode(X_test), encode(X_cal), encode(X_zeroday)
print(f" A_train {A_train.shape}  A_test {A_test.shape}  A_cal {A_cal.shape}  A_zeroday {A_zeroday.shape}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
plot_class_balance_bars(y_train, class_names, title="Train (subset) class balance", ax=ax[0])
plot_class_balance_pie(y_train, class_names, title="Train (subset) class balance", ax=ax[1])
plt.tight_layout(); savefig("part2_class_balance")

LOADING RAW DATA SPLITS
 train(full): (151049, 10)  test: (18883, 10)  cal: (18883, 10)  zeroday: (10984, 10)
 train(subset): (20156, 10) via greedy_dpp
 A_train (20156, 6)  A_test (18883, 6)  A_cal (18883, 6)  A_zeroday (10984, 6)
  saved figure -> /kaggle/working/quantum-sentinel/figures/part2_class_balance.png


In [6]:
# ============================================================
# PART 3 — Quantum device + ForwardCache (ONE forward pass per split, ever)
# ============================================================
print("="*60); print("QUANTUM CIRCUIT"); print("="*60)

dev = create_quantum_device(num_qubits)                 # default.mixed
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers,
                                         noise_rate=noise_rate, reupload=reupload)
theta_device = theta_star.to(DEVICE)
print(f" device={DEVICE}  wires={num_qubits}  theta on {theta_device.device}")

CACHE_LATEST = CACHE_DIR / "forward_cache_latest.pt"
cache = ForwardCache(store_device="cpu")

splits = [("train", A_train), ("test", A_test), ("cal", A_cal), ("zeroday", A_zeroday)]

if CACHE_LATEST.exists():
    print(f"RESUME: loading forward cache from {CACHE_LATEST}")
    cache_data = torch.load(CACHE_LATEST, map_location="cpu", weights_only=False)
    for key, _ in splits:
        cache._entries[key] = {
            "z": cache_data[f"{key}_z"], "rho": cache_data[f"{key}_rho"],
            "n": cache_data[f"{key}_n"], "p": noise_rate, "X": cache_data[f"{key}_X"],
        }
else:
    print("BUILDING FORWARD CACHE (once)")
    t0 = time.time()
    for key, A in splits:
        entry = run_batched_safely(cache.compute, key, A, theta_device, forward_circuit,
                                    device=DEVICE, batch_size=128, min_batch=8, p=noise_rate,
                                    label=f"cache[{key}]")
        mb = (entry["rho"].numel()*entry["rho"].element_size() +
              entry["z"].numel()*entry["z"].element_size()) / 1e6
        print(f" [{key:8s}] n={entry['n']:>7,} rho={tuple(entry['rho'].shape)} "
              f"z={tuple(entry['z'].shape)}  {mb:.1f} MB")
    print(f"total: {time.time()-t0:.1f}s")

    save_dict = {"noise_rate": noise_rate, "num_layers": num_layers, "num_qubits": num_qubits}
    for key, _ in splits:
        entry = cache.get(key)
        save_dict[f"{key}_z"]   = entry["z"]
        save_dict[f"{key}_rho"] = entry["rho"]
        save_dict[f"{key}_n"]   = entry["n"]
        save_dict[f"{key}_X"]   = entry["X"]
    torch.save(save_dict, CACHE_LATEST)
    print(f"saved cache -> {CACHE_LATEST} ({CACHE_LATEST.stat().st_size/1e6:.1f} MB)")

for row in cache.memory_report():
    print(f"  {row['key']:8s}: n={row['n_samples']:5d}  {row['MB']:7.1f} MB (CPU)")

QUANTUM CIRCUIT
 device=cpu  wires=6  theta on cpu
BUILDING FORWARD CACHE (once)
 [train   ] n= 20,156 rho=(20156, 64, 64) z=(20156, 6)  1321.4 MB
 [test    ] n= 18,883 rho=(18883, 64, 64) z=(18883, 6)  1238.0 MB
 [cal     ] n= 18,883 rho=(18883, 64, 64) z=(18883, 6)  1238.0 MB
 [zeroday ] n= 10,984 rho=(10984, 64, 64) z=(10984, 6)  720.1 MB
total: 343.5s
saved cache -> /kaggle/working/quantum-sentinel/caches/forward_cache_latest.pt (4519.9 MB)
  train   : n=20156   1321.4 MB (CPU)
  test    : n=18883   1238.0 MB (CPU)
  cal     : n=18883   1238.0 MB (CPU)
  zeroday : n=10984    720.1 MB (CPU)


In [7]:
# ============================================================
# PART 4 — Test-set classification report (cached z -> head)
# ============================================================
y_true_test, y_pred_test = cached_predict_labels(cache.get("test"), y_test, classifier_head,
                                                   device=DEVICE, batch_size=256)
print(classification_report(y_true_test, y_pred_test, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true_test, y_pred_test, labels=range(num_classes))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion matrix (test)"); plt.xticks(rotation=60); plt.tight_layout()
savefig("part4_confusion_matrix")

savejson({"classification_report": classification_report(y_true_test, y_pred_test,
          target_names=class_names, zero_division=0, output_dict=True)}, "part4_test_eval")

                         precision    recall  f1-score   support

       Backdoor_Malware       0.06      0.44      0.11         9
          BenignTraffic       0.64      0.37      0.47       458
       BrowserHijacking       0.00      0.00      0.00        16
       CommandInjection       0.02      0.08      0.04        13
 DDoS-ACK_Fragmentation       0.78      0.75      0.77       120
        DDoS-HTTP_Flood       0.12      0.50      0.20        70
        DDoS-ICMP_Flood       0.84      0.98      0.90      3025
DDoS-ICMP_Fragmentation       0.81      0.94      0.87       190
      DDoS-PSHACK_Flood       0.88      0.54      0.67      1712
       DDoS-RSTFINFlood       0.68      0.91      0.78      1688
         DDoS-SYN_Flood       0.62      0.91      0.74      1706
         DDoS-SlowLoris       0.41      0.45      0.43        62
DDoS-SynonymousIP_Flood       0.71      0.56      0.63      1509
         DDoS-TCP_Flood       0.75      0.69      0.72      1892
         DDoS-UDP_Flood 

PosixPath('/kaggle/working/quantum-sentinel/logs/part4_test_eval.json')

In [8]:
# ============================================================
# PART 5 — Global + class-conditional conformal calibration (from cache)
# ============================================================
print(f"min calibration size for alpha={DEFAULT_ALPHA}: n >= {min_calibration_size(DEFAULT_ALPHA):.1f} "
      f"(have n={len(A_cal)})")

q_final, cal_scores_sorted = cached_calibrate_threshold(
    cache.get("cal"), prototypes, alpha=DEFAULT_ALPHA, device=DEVICE, batch_size=256)
print(f"Calibrated global threshold q = {q_final:.4f}")

requested_alphas = (0.01, 0.05, 0.1, 0.2)
feasible_alphas = [a for a in requested_alphas if len(A_cal) >= min_calibration_size(a)]
skipped_alphas = [a for a in requested_alphas if a not in feasible_alphas]
if skipped_alphas:
    print(f"Skipping alpha(s) {skipped_alphas}: n={len(A_cal)} too small (Proposition 3 abstention).")

alpha_rows = cached_conformal_alpha_sweep(cache.get("cal"), cache.get("test"), prototypes,
                                           alphas=feasible_alphas, device=DEVICE, batch_size=256) \
             if feasible_alphas else []
df_alpha = pd.DataFrame(alpha_rows)
auroc_exch = two_sample_discriminability_auroc(A_cal, A_test, seed=SEED)
print(f"cal-vs-test exchangeability AUROC: {auroc_exch:.3f} (want ~0.50)")

if not df_alpha.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(df_alpha["alpha"], df_alpha["empirical_false_alarm_rate"], marker="o", label="empirical FAR")
    ax.plot(df_alpha["alpha"], df_alpha["alpha"], "--", color="black", label="target = alpha")
    ax.set_xlabel("alpha"); ax.set_ylabel("false-alarm rate")
    ax.set_title(f"Proposition 3 alpha-sweep (AUROC={auroc_exch:.3f})"); ax.legend()
    plt.tight_layout(); savefig("part5_alpha_sweep")
savecsv(df_alpha, "part5_alpha_sweep")

# --- Mondrian (per-class) calibration ---
q_by_class, calib_meta = class_conditional_calibrate(
    theta_device, A_cal, y_cal, prototypes, forward_circuit,
    alpha=DEFAULT_ALPHA, device=DEVICE, batch_size=8, fallback="global",
)
for c in sorted(prototypes.keys()):
    m = calib_meta[c]
    print(f" class {c} ({class_names[c]:20s}): n={m['n']:5d} q={m['q']:.4f} status={m['status']}")
print(f" global (marginal) q = {calib_meta['_global']['q']:.4f}")

# inline save for crash insurance right after the ~7h Mondrian block
savejson({
    "q_final": float(q_final),
    "q_by_class": {int(k): float(v) for k, v in q_by_class.items()},
    "calib_meta": to_jsonable(calib_meta),
}, "part5_q_by_class")
print("  saved part5_q_by_class.json (inline checkpoint after Mondrian calibration)")

q_global_map = {c: q_final for c in prototypes}
far_global   = per_class_empirical_far(theta_device, A_test, y_test, prototypes, forward_circuit,
                                        q_global_map, device=DEVICE, batch_size=8)
far_perclass = per_class_empirical_far(theta_device, A_test, y_test, prototypes, forward_circuit,
                                        q_by_class, device=DEVICE, batch_size=8)
df_far = pd.DataFrame([
    {"class": class_names[r["class"]], "n": r["n"],
     "FAR_global": r["empirical_far"], "FAR_per_class": r2["empirical_far"]}
    for r, r2 in zip(far_global, far_perclass)
])
print(df_far)
savecsv(df_far, "part5_per_class_far")

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_far)); w = 0.35
ax.bar(x - w/2, df_far["FAR_global"], w, label="global", color="steelblue")
ax.bar(x + w/2, df_far["FAR_per_class"], w, label="per-class", color="darkorange")
ax.axhline(DEFAULT_ALPHA, color="black", linestyle="--", label=f"target alpha={DEFAULT_ALPHA}")
ax.set_xticks(x); ax.set_xticklabels(df_far["class"], rotation=60, ha="right")
ax.legend(); plt.tight_layout(); savefig("part5_mondrian_far")

safe_empty_cache()

min calibration size for alpha=0.05: n >= 19.0 (have n=18883)
Calibrated global threshold q = 0.2447
cal-vs-test exchangeability AUROC: 0.498 (want ~0.50)
  saved figure -> /kaggle/working/quantum-sentinel/figures/part5_alpha_sweep.png
  saved table -> /kaggle/working/quantum-sentinel/tables/part5_alpha_sweep.csv
 class 0 (Backdoor_Malware    ): n=    9 q=0.2447 status=insufficient for alpha=0.05 -- used GLOBAL threshold (Calibration set too small for alpha=0.05: n=9, but Proposition 3 requires n >= (1/alpha) - 1 = 19.00. Abstaining rather than using the largest calibration score -- collect more calibration data or increase alpha.)
 class 1 (BenignTraffic       ): n=  458 q=0.4049 status=ok
 class 2 (BrowserHijacking    ): n=   16 q=0.2447 status=insufficient for alpha=0.05 -- used GLOBAL threshold (Calibration set too small for alpha=0.05: n=16, but Proposition 3 requires n >= (1/alpha) - 1 = 19.00. Abstaining rather than using the largest calibration score -- collect more calibration

In [9]:
# ============================================================
# DAY 15 — Depolarizing-channel contraction check on REAL cached rho(x)
# ============================================================
prop1_records = check_proposition1_real_data(
    cache.get("test")["rho"], p_values=(0.0, 0.01, 0.1, 0.3, 0.5, 0.7, 1.0),
    n_pairs=25, seed=SEED,
)
max_resid = assert_proposition1(prop1_records, tol=PROP1_RESIDUAL_TOL)
print(f"Proposition 1 PASSED on real data: max residual = {max_resid:.3e}")
savejson({"max_resid": float(max_resid)}, "day15_max_resid")

df_prop1 = pd.DataFrame(prop1_records)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(df_prop1["D_tr_expected"], df_prop1["D_tr_noisy"], s=10, alpha=0.4)
lims = [0, df_prop1["D_tr_expected"].max() * 1.05]
ax[0].plot(lims, lims, "r--", label="y = x"); ax[0].legend()
ax[0].set_title("Proposition 1: measured vs theoretical")
err_by_p = df_prop1.groupby("p")["residual"].max()
ax[1].plot(err_by_p.index, err_by_p.values, marker="o", color="darkred")
ax[1].set_yscale("log"); ax[1].set_title("Residual vs p (log)")
plt.tight_layout(); savefig("day15_proposition1")
savecsv(df_prop1, "day15_proposition1")

Proposition 1 PASSED on real data: max residual = 9.992e-16
  saved json -> /kaggle/working/quantum-sentinel/logs/day15_max_resid.json
  saved figure -> /kaggle/working/quantum-sentinel/figures/day15_proposition1.png
  saved table -> /kaggle/working/quantum-sentinel/tables/day15_proposition1.csv


PosixPath('/kaggle/working/quantum-sentinel/tables/day15_proposition1.csv')

In [10]:
# ============================================================
# DAY 16 — Analytic L_phi (Lemma 1) + sampled-ratio tightness (from cache)
# ============================================================
L_PHI_ANALYTIC = analytic_lipschitz_bound(num_layers, reupload=reupload)
print(f"Analytic Lipschitz bound: L_phi <= R/2 = {num_layers}/2 = {L_PHI_ANALYTIC:.4f}")

lipschitz_diag = {}
for name in ["train", "test", "cal", "zeroday"]:
    diag = cached_lipschitz_percentile(cache.get(name), n_pairs=min(300, cache.get(name)["n"]//2),
                                        seed=SEED, percentile=95)
    lipschitz_diag[name] = {**diag, **check_lipschitz_tightness(diag, L_PHI_ANALYTIC)}
    status = "OK" if lipschitz_diag[name]["within_bound"] else "!! BOUND VIOLATED !!"
    print(f"{name:8s}: max ratio={lipschitz_diag[name]['max_sampled_ratio']:.4f}  "
          f"gap={lipschitz_diag[name]['tightness_gap']:.4f}  [{status}]")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, res) in zip(axes.flat, lipschitz_diag.items()):
    if len(res["ratios"]) == 0:
        ax.set_title(f"{name}: no data"); continue
    ax.hist(res["ratios"], bins=30, alpha=0.8)
    ax.axvline(L_PHI_ANALYTIC, color="red", linestyle="--", label=f"bound={L_PHI_ANALYTIC:.3f}")
    ax.axvline(res["max"], color="black", linestyle=":", label=f"max={res['max']:.3f}")
    ax.set_title(f"L_phi tightness — {name}"); ax.legend()
plt.tight_layout(); savefig("day16_lipschitz_tightness")

savejson({n: {k: v for k, v in r.items() if k != "ratios"} for n, r in lipschitz_diag.items()},
         "day16_lipschitz")

Analytic Lipschitz bound: L_phi <= R/2 = 3/2 = 1.5000
train   : max ratio=0.8792  gap=0.6208  [OK]
test    : max ratio=0.8776  gap=0.6224  [OK]
cal     : max ratio=0.8814  gap=0.6186  [OK]
zeroday : max ratio=0.8858  gap=0.6142  [OK]
  saved figure -> /kaggle/working/quantum-sentinel/figures/day16_lipschitz_tightness.png
  saved json -> /kaggle/working/quantum-sentinel/logs/day16_lipschitz.json


PosixPath('/kaggle/working/quantum-sentinel/logs/day16_lipschitz.json')

In [11]:
# ============================================================
# Mondrian Proposition-2: per-class F_in_c / F_out_c / Delta_c / eps*_c
# (needs L_PHI_ANALYTIC from Day 16 — run in this order, not before)
# ============================================================
class_ids = sorted(prototypes.keys())
rho_test_all = cache.get("test")["rho"]
rho_zday_all = cache.get("zeroday")["rho"]

per_class_prop2_rows = []
for c in class_ids:
    proto_c = prototypes[c]
    mask_c = (y_test == c); n_c = int(mask_c.sum())
    with torch.no_grad():
        rho_c_test = rho_test_all[mask_c].to(DEVICE)
        proto_b = proto_c.unsqueeze(0).expand(rho_c_test.shape[0], -1, -1)
        f_own_c = fidelity_pairwise(rho_c_test, proto_b).float().cpu().numpy()
        F_in_c = float(f_own_c.min()) if n_c > 0 else float("nan")

        rho_z = rho_zday_all.to(DEVICE)
        proto_b_z = proto_c.unsqueeze(0).expand(rho_z.shape[0], -1, -1)
        f_zday_c = fidelity_pairwise(rho_z, proto_b_z).float().cpu().numpy()
        F_out_c = float(f_zday_c.max())

    delta_c, eps_star_c = proposition2_epsilon_star(F_in_c, F_out_c, p=noise_rate, L_phi=L_PHI_ANALYTIC)
    per_class_prop2_rows.append({"class": class_names[c], "n_known": n_c,
                                  "F_in_c": F_in_c, "F_out_c": F_out_c,
                                  "Delta_c": delta_c, "epsilon_star_c": eps_star_c})
    safe_empty_cache()

df_prop2_perclass = pd.DataFrame(per_class_prop2_rows)
df_prop2_perclass["separable"] = df_prop2_perclass["Delta_c"] > 0
print(df_prop2_perclass.to_string(index=False))
savecsv(df_prop2_perclass, "day18c_mondrian_proposition2")

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["seagreen" if d > 0 else "firebrick" for d in df_prop2_perclass["Delta_c"]]
ax.bar(df_prop2_perclass["class"], df_prop2_perclass["Delta_c"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); savefig("day18c_mondrian_delta")

                  class  n_known   F_in_c  F_out_c   Delta_c  epsilon_star_c  separable
       Backdoor_Malware        9 0.451121 0.511216 -0.403678       -0.135919      False
          BenignTraffic      458 0.466568 0.496800 -0.381286       -0.128379      False
       BrowserHijacking       16 0.487997 0.505268 -0.378114       -0.127311      False
       CommandInjection       13 0.503572 0.500293 -0.364247       -0.122642      False
 DDoS-ACK_Fragmentation      120 0.586448 0.970407 -0.780394       -0.262759      False
        DDoS-HTTP_Flood       70 0.579854 0.903665 -0.718386       -0.241881      False
        DDoS-ICMP_Flood     3025 0.368371 0.931722 -0.861400       -0.290034      False
DDoS-ICMP_Fragmentation      190 0.398202 0.965486 -0.882784       -0.297234      False
      DDoS-PSHACK_Flood     1712 0.567023 0.856248 -0.679950       -0.228939      False
       DDoS-RSTFINFlood     1688 0.513838 0.834654 -0.692541       -0.233179      False
         DDoS-SYN_Flood     1706

In [12]:
# ============================================================
# DAY 17 — Certified radius R = m / (2(1-p) L_phi C_f), from cache
# ============================================================
labels_test_final, radii_test_final, scores_test_final, fmaps_test_final = cached_qsnet_infer(
    cache.get("test"), prototypes, q_final, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256,
)
correct_mask = (labels_test_final == y_test)
certified_radii_correct = radii_test_final[correct_mask]
print(f"Correctly-classified & accepted: {correct_mask.sum()} / {len(y_test)}")
print(f"Certified radius: mean={certified_radii_correct.mean():.4f} "
      f"median={np.median(certified_radii_correct):.4f} "
      f"min={certified_radii_correct.min():.4f} max={certified_radii_correct.max():.4f}")

torch.save({
    "y_test": y_test,
    "labels_test_final": labels_test_final,
    "radii_test_final": radii_test_final,
    "scores_test_final": scores_test_final,
    "certified_radii_correct": certified_radii_correct,
}, CKPT_DIR / "day17_test_infer.pt")

plt.figure(figsize=(7, 4.5))
plt.hist(certified_radii_correct, bins=40, color="teal", alpha=0.8)
plt.axvline(certified_radii_correct.mean(), color="red", linestyle="--",
            label=f"mean={certified_radii_correct.mean():.3f}")
plt.legend(); plt.title("Certified-radius distribution (test set)")
plt.tight_layout(); savefig("day17_certified_radius")

Correctly-classified & accepted: 12599 / 18883
Certified radius: mean=0.0154 median=0.0144 min=0.0000 max=0.1005
  saved figure -> /kaggle/working/quantum-sentinel/figures/day17_certified_radius.png


In [13]:
# ============================================================
# DAY 18 — Proposition 2: F_in, F_out, Delta, epsilon*
# ============================================================
F_max_test    = cached_f_max(cache.get("test"), prototypes, device=DEVICE, batch_size=256)
F_max_zeroday = cached_f_max(cache.get("zeroday"), prototypes, device=DEVICE, batch_size=256)

F_IN  = worst_case_f_in(F_max_test, percentile=0.0)
F_OUT = float(F_max_zeroday.max())
delta_strict, eps_star_strict = proposition2_epsilon_star(F_IN, F_OUT, p=noise_rate, L_phi=L_PHI_ANALYTIC)
F_out_beta, delta_beta, eps_beta = proposition2_epsilon_beta(
    F_IN, F_max_zeroday, L_phi=L_PHI_ANALYTIC, p=noise_rate, beta=DEFAULT_BETA)

print(f"F_in={F_IN:.4f}  F_out={F_OUT:.4f}")
print(f"Delta(strict)={delta_strict:+.4f}  eps*(strict)={eps_star_strict:.4f}"
      f"{' [VACUOUS]' if eps_star_strict <= 0 else ''}")
print(f"F_out^beta={F_out_beta:.4f}  Delta^beta={delta_beta:+.4f}  eps^beta={eps_beta:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(F_max_test, bins=30, alpha=0.6, label="F_max, known", color="seagreen")
ax[0].hist(F_max_zeroday, bins=30, alpha=0.6, label="F_max, zero-day", color="firebrick")
ax[0].axvline(F_IN, color="green", linestyle="--"); ax[0].axvline(F_OUT, color="red", linestyle="--")
ax[0].legend(fontsize=8); ax[0].set_title("F_in vs F_out")
bar_vals = [max(eps_star_strict, 0), max(eps_beta, 0)]
ax[1].bar(["strict", f"beta={DEFAULT_BETA}"], bar_vals,
          color=["slateblue" if eps_star_strict>0 else "lightgray",
                 "darkorange" if eps_beta>0 else "lightgray"])
ax[1].set_title("Certified separable budget epsilon*")
plt.tight_layout(); savefig("day18_proposition2")

F_in=0.4880  F_out=0.9769
Delta(strict)=-0.8498  eps*(strict)=-0.2861 [VACUOUS]
F_out^beta=0.9655  Delta^beta=-0.8384  eps^beta=-0.2823
  saved figure -> /kaggle/working/quantum-sentinel/figures/day18_proposition2.png


In [14]:
# ============================================================
# DAY 18b — Doubly-robust F_in / F_out (percentile-based)
# ============================================================
F_in_robust, F_out_robust, delta_robust, eps_robust = proposition2_epsilon_robust(
    F_max_test, F_max_zeroday, p=noise_rate, L_phi=L_PHI_ANALYTIC,
    lower_percentile=DEFAULT_LOWER_PERCENTILE, upper_percentile=DEFAULT_UPPER_PERCENTILE,
)
df_eps_compare = pd.DataFrame([
    {"variant": "strict", "F_in": F_IN, "F_out": F_OUT, "Delta": delta_strict, "epsilon": eps_star_strict},
    {"variant": f"F_out-relaxed(beta={DEFAULT_BETA})", "F_in": F_IN, "F_out": F_out_beta,
     "Delta": delta_beta, "epsilon": eps_beta},
    {"variant": "doubly-robust(5/97pct)", "F_in": F_in_robust, "F_out": F_out_robust,
     "Delta": delta_robust, "epsilon": eps_robust},
])
print(df_eps_compare.to_string(index=False))
savecsv(df_eps_compare, "day18b_epsilon_variants")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["slateblue" if e > 0 else "lightgray" for e in df_eps_compare["epsilon"]]
ax.bar(df_eps_compare["variant"], df_eps_compare["epsilon"].clip(lower=0), color=colors)
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); savefig("day18b_epsilon_variants")

                 variant     F_in    F_out     Delta   epsilon
                  strict 0.487997 0.976947 -0.849792 -0.286125
F_out-relaxed(beta=0.05) 0.487997 0.965546 -0.838391 -0.282286
  doubly-robust(5/97pct) 0.757267 0.966404 -0.619510 -0.208589
  saved table -> /kaggle/working/quantum-sentinel/tables/day18b_epsilon_variants.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day18b_epsilon_variants.png


In [15]:
# ============================================================
# PART 1 HANDOFF — freeze all state for part 2 (Day 19+)
# ============================================================
PART1_HANDOFF = CKPT_DIR / "part1_handoff.pt"

part1_handoff = {
    "version": 1,
    "dataset": DATASET,
    "seed": SEED,
    "noise_rate": noise_rate,
    "num_qubits": num_qubits,
    "num_layers": num_layers,
    "reupload": reupload,
    "default_alpha": DEFAULT_ALPHA,
    "default_cf": DEFAULT_CF,
    "zero_day": ZERO_DAY,
    "class_names": class_names,
    "checkpoint_path": str(CHECKPOINT_PATH),

    # PART 5
    "q_final": float(q_final),
    "q_by_class": {int(k): float(v) for k, v in q_by_class.items()},
    "calib_meta": to_jsonable(calib_meta),

    # Day 15–16
    "L_PHI_ANALYTIC": float(L_PHI_ANALYTIC),
    "max_resid": float(max_resid),

    # Day 17 (the expensive one — save it!)
    "y_test": y_test,
    "labels_test_final": labels_test_final,
    "radii_test_final": radii_test_final,
    "scores_test_final": scores_test_final,
    "certified_radii_correct": certified_radii_correct,

    # Day 18–18b
    "F_max_test": F_max_test,
    "F_max_zeroday": F_max_zeroday,
    "F_IN": float(F_IN),
    "F_OUT": float(F_OUT),
    "delta_strict": float(delta_strict),
    "eps_star_strict": float(eps_star_strict),
    "F_out_beta": float(F_out_beta),
    "delta_beta": float(delta_beta),
    "eps_beta": float(eps_beta),
    "F_in_robust": float(F_in_robust),
    "F_out_robust": float(F_out_robust),
    "delta_robust": float(delta_robust),
    "eps_robust": float(eps_robust),
}

torch.save(part1_handoff, PART1_HANDOFF)
print(f"saved part1 handoff -> {PART1_HANDOFF} ({PART1_HANDOFF.stat().st_size/1e6:.2f} MB)")

# human-readable copies
savejson({"q_final": q_final, "q_by_class": part1_handoff["q_by_class"]}, "part5_q_by_class")
savejson({
    "F_IN": F_IN, "F_OUT": F_OUT,
    "eps_star_strict": eps_star_strict, "eps_beta": eps_beta, "eps_robust": eps_robust,
    "L_PHI_ANALYTIC": L_PHI_ANALYTIC, "max_resid": max_resid,
    "certified_radius_mean": float(certified_radii_correct.mean()),
}, "part1_scalars")

savejson({
    "version": 1,
    "dataset": DATASET,
    "completed_through": "DAY 18b",
    "checkpoints": ["checkpoints/part1_handoff.pt", "checkpoints/day17_test_infer.pt"],
    "forward_cache": "caches/forward_cache_latest.pt",
    "tables": [
        "part5_alpha_sweep", "part5_per_class_far",
        "day15_proposition1", "day18c_mondrian_proposition2", "day18b_epsilon_variants",
    ],
    "logs": ["part4_test_eval", "day16_lipschitz", "part5_q_by_class", "part1_scalars", "part1_manifest", "day15_max_resid"],
}, "part1_manifest")

print("=" * 60)
print("PART 1 COMPLETE")
print("Attach this notebook's OUTPUT as a Kaggle dataset for part 2.")
print("=" * 60)

saved part1 handoff -> /kaggle/working/quantum-sentinel/checkpoints/part1_handoff.pt (0.78 MB)
  saved json -> /kaggle/working/quantum-sentinel/logs/part5_q_by_class.json
  saved json -> /kaggle/working/quantum-sentinel/logs/part1_scalars.json
  saved json -> /kaggle/working/quantum-sentinel/logs/part1_manifest.json
PART 1 COMPLETE
Attach this notebook's OUTPUT as a Kaggle dataset for part 2.
